# Vintern-1B-v3_5 — official FP16 T4 audit

Upload this notebook to Kaggle, attach dataset `thvu165/aic-2026-keyframes`, select **GPU T4 x2**, enable Internet, then choose **Run All**. The notebook deliberately masks GPU 1 and benchmarks only GPU 0. It does not publish, commit, or upload any artifact.

The run downloads the official revision, verifies the exact `model.safetensors` SHA-256, uses the pinned EasyOCR CRAFT detector to extract 100 real text crops, and measures FP16 load/inference VRAM and latency for `max_num=1/2/6`. Results are written to `/kaggle/working/vintern_t4_fp16_benchmark.json`.

In [ ]:
# Keep this cell first: hide the second T4 before importing torch.
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
os.environ["TOKENIZERS_PARALLELISM"] = "false"

import importlib.metadata as md
import importlib.util
import subprocess
import sys

def version_or_missing(name):
    try:
        return md.version(name)
    except md.PackageNotFoundError:
        return None

protected_packages = ["torch", "torchvision", "numpy", "opencv-python", "opencv-python-headless", "scikit-image", "pandas"]
protected_before = {name: version_or_missing(name) for name in protected_packages}
python_only_runtime = [
    "transformers==4.43.4",
    "tokenizers==0.19.1",
    "huggingface-hub==0.24.7",
    "easyocr==1.7.2",
    "python-bidi==0.6.6",
    "pyclipper==1.3.0.post6",
    "ninja==1.11.1.4",
    "sentencepiece==0.2.0",
]
# Install the model-specific Python/Rust wheels without dependency resolution.
# Kaggle already supplies the compatible binary stack (NumPy/OpenCV/scikit-image/
# pandas/Torch). Transformers 4.42.1 declares NumPy <2; 4.43.4 is the nearest
# compatible release for Kaggle's NumPy 2.0 stack. Normal dependency resolution
# would still be allowed to rewrite unrelated packages, so keep it disabled.
subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", "--no-cache-dir", "--no-deps", *python_only_runtime])
required_preinstalled = ["accelerate", "safetensors", "timm", "einops", "yaml", "scipy", "skimage", "shapely"]
missing_preinstalled = [name for name in required_preinstalled if importlib.util.find_spec(name) is None]
assert not missing_preinstalled, ("Kaggle image is missing expected runtime modules", missing_preinstalled)
protected_after = {name: version_or_missing(name) for name in protected_packages}
assert protected_before == protected_after, ("pip changed Kaggle binary packages", protected_before, protected_after)
abi_probe = subprocess.run(
    [sys.executable, "-c", "import numpy, cv2, pandas, skimage; print(numpy.__version__, cv2.__version__, pandas.__version__, skimage.__version__)"],
    text=True, capture_output=True,
)
assert abi_probe.returncode == 0, "Kaggle binary ABI is already dirty; stop the session completely and start a fresh one.\n" + abi_probe.stderr
print("ABI_PROBE", abi_probe.stdout.strip())
print({"protected_packages": protected_after, "transformers": version_or_missing("transformers"), "tokenizers": version_or_missing("tokenizers"), "huggingface-hub": version_or_missing("huggingface-hub"), "easyocr": version_or_missing("easyocr")})

In [ ]:
from __future__ import annotations

import hashlib
import json
import math
import platform
import shutil
import statistics
import time
import urllib.request
import zipfile
from datetime import datetime, timezone
from pathlib import Path

import cv2
import numpy as np
import torch
from PIL import Image

MODEL_ID = "5CD-AI/Vintern-1B-v3_5"
MODEL_REVISION = "b98f263eab246eb5269ade64edbdca8a887dc44d"
MODEL_WEIGHT_SHA256 = "296a16a6bf28e6d3f0fb9298deba70b3cfa1d7519f4aa326e2f862bf2e63be05"
MODEL_WEIGHT_BYTES = 3_752_849_256
MODEL_REQUIRED_GIT_OIDS = {
    "config.json": "2668519f652eddcc2abbb56a52518d38c4f88887",
    "configuration_intern_vit.py": "7e630c456eb9cf350e55bf850c3ff72f445a7e17",
    "configuration_internvl_chat.py": "799209432caf749e77de4c889ec03fe6a32fcdf9",
    "conversation.py": "76fcea7f331de42b6aed7e39fdf80728f9784b7f",
    "modeling_intern_vit.py": "1c5c043a4b860720b3b6e55107e8e6ecf0c573de",
    "modeling_internvl_chat.py": "41f48cd5cb907e025725fc42ac819cf3f03c01b5",
}
CRAFT_URL = "https://github.com/JaidedAI/EasyOCR/releases/download/pre-v1.1.6/craft_mlt_25k.zip"
CRAFT_ZIP_SHA256 = "8dc6a1c703a89ed56308ef742d26ebd45c656248cbbbda6e7fe60e569f873e65"
CRAFT_WEIGHT_SHA256 = "4a5efbfb48b4081100544e75e1e2b57f8de3d84f213004b14b85fd4b3748db17"
TARGET_CROPS = 100
MAX_SOURCE_IMAGES = 1_000
MAX_CROPS_PER_IMAGE = 3
TILE_MODES = (1, 2, 6)
MAX_NEW_TOKENS = 96
WORK_DIR = Path("/kaggle/working/vintern-t4-audit")
CROP_DIR = WORK_DIR / "craft-crops"
RESULT_PATH = Path("/kaggle/working/vintern_t4_fp16_benchmark.json")
MANIFEST_PATH = Path("/kaggle/working/vintern_craft_crops_manifest.jsonl")
WORK_DIR.mkdir(parents=True, exist_ok=True)
CROP_DIR.mkdir(parents=True, exist_ok=True)

def utc_now():
    return datetime.now(timezone.utc).isoformat()

def sha256_file(path: Path, chunk_size: int = 8 * 1024 * 1024) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        while chunk := handle.read(chunk_size):
            digest.update(chunk)
    return digest.hexdigest()

def git_blob_oid(path: Path, chunk_size: int = 8 * 1024 * 1024) -> str:
    digest = hashlib.sha1()
    digest.update(f"blob {path.stat().st_size}\0".encode("ascii"))
    with path.open("rb") as handle:
        while chunk := handle.read(chunk_size):
            digest.update(chunk)
    return digest.hexdigest()

def package_versions():
    names = ["torch", "torchvision", "transformers", "easyocr", "accelerate", "safetensors", "timm", "einops", "opencv-python", "numpy", "Pillow"]
    return {name: version_or_missing(name) for name in names}

def gpu_query():
    fields = "index,name,memory.total,memory.used,driver_version"
    output = subprocess.check_output(["nvidia-smi", f"--query-gpu={fields}", "--format=csv,noheader,nounits"], text=True)
    return [line.strip() for line in output.splitlines() if line.strip()]

assert torch.cuda.is_available(), "CUDA is unavailable; select Kaggle GPU T4 x2 first"
assert torch.cuda.device_count() == 1, f"Expected exactly one visible GPU after masking, got {torch.cuda.device_count()}"
gpu_name = torch.cuda.get_device_name(0)
assert "T4" in gpu_name.upper(), f"Expected T4, got {gpu_name}"
print("VISIBLE_GPU", gpu_name, torch.cuda.get_device_properties(0).total_memory)
print("NVIDIA_SMI_ALL_PHYSICAL_GPUS", gpu_query())
print("PACKAGES", package_versions())

In [ ]:
# Download and verify the official CRAFT detector bytes used by EasyOCR.
import easyocr

easyocr_model_dir = WORK_DIR / "easyocr-models"
easyocr_model_dir.mkdir(parents=True, exist_ok=True)
craft_zip = WORK_DIR / "craft_mlt_25k.zip"
craft_weight = easyocr_model_dir / "craft_mlt_25k.pth"
if not craft_zip.exists() or sha256_file(craft_zip) != CRAFT_ZIP_SHA256:
    urllib.request.urlretrieve(CRAFT_URL, craft_zip)
assert sha256_file(craft_zip) == CRAFT_ZIP_SHA256, "CRAFT ZIP checksum mismatch"
with zipfile.ZipFile(craft_zip) as archive:
    member = next(name for name in archive.namelist() if name.endswith("craft_mlt_25k.pth"))
    with archive.open(member) as source, craft_weight.open("wb") as target:
        shutil.copyfileobj(source, target)
assert sha256_file(craft_weight) == CRAFT_WEIGHT_SHA256, "CRAFT weight checksum mismatch"

reader = easyocr.Reader(
    ["vi", "en"],
    gpu="cuda:0",
    model_storage_directory=str(easyocr_model_dir),
    download_enabled=False,
    detector=True,
    recognizer=False,
    verbose=True,
)

def iter_input_images(root: Path):
    extensions = {".jpg", ".jpeg", ".png", ".webp"}
    for dirpath, dirnames, filenames in os.walk(root):
        dirnames.sort()
        for filename in sorted(filenames):
            path = Path(dirpath) / filename
            if path.suffix.lower() in extensions:
                yield path

def clamp_box(box, width, height, padding=4):
    x1, y1, x2, y2 = box
    return (max(0, int(math.floor(x1)) - padding), max(0, int(math.floor(y1)) - padding), min(width, int(math.ceil(x2)) + padding), min(height, int(math.ceil(y2)) + padding))

input_root = Path("/kaggle/input")
all_roots = sorted(path for path in input_root.iterdir() if path.is_dir())
assert all_roots, "No Kaggle input dataset is attached"
print("INPUT_ROOTS", [str(path) for path in all_roots])

crop_records = []
scanned = 0
detect_started = time.perf_counter()
for source_path in iter_input_images(input_root):
    if scanned >= MAX_SOURCE_IMAGES or len(crop_records) >= TARGET_CROPS:
        break
    scanned += 1
    image_bgr = cv2.imread(str(source_path), cv2.IMREAD_COLOR)
    if image_bgr is None:
        continue
    height, width = image_bgr.shape[:2]
    scale = min(1.0, 1600.0 / max(height, width))
    if scale < 1.0:
        image_bgr = cv2.resize(image_bgr, (round(width * scale), round(height * scale)), interpolation=cv2.INTER_AREA)
    height, width = image_bgr.shape[:2]
    horizontal, free = reader.detect(
        image_bgr, min_size=10, text_threshold=0.6, low_text=0.3, link_threshold=0.3,
        canvas_size=2560, mag_ratio=1.0, slope_ths=0.1, ycenter_ths=0.5,
        height_ths=0.5, width_ths=0.5, add_margin=0.1, reformat=True,
    )
    boxes = []
    for item in (horizontal[0] if horizontal else []):
        x1, x2, y1, y2 = item
        boxes.append((x1, y1, x2, y2))
    for polygon in (free[0] if free else []):
        points = np.asarray(polygon, dtype=np.float32)
        boxes.append((float(points[:, 0].min()), float(points[:, 1].min()), float(points[:, 0].max()), float(points[:, 1].max())))
    boxes = sorted(boxes, key=lambda box: (box[2] - box[0]) * (box[3] - box[1]), reverse=True)[:MAX_CROPS_PER_IMAGE]
    for raw_box in boxes:
        x1, y1, x2, y2 = clamp_box(raw_box, width, height)
        if x2 - x1 < 12 or y2 - y1 < 8 or (x2 - x1) * (y2 - y1) < 160:
            continue
        crop_bgr = image_bgr[y1:y2, x1:x2]
        crop_index = len(crop_records)
        crop_path = CROP_DIR / f"crop-{crop_index:03d}.jpg"
        assert cv2.imwrite(str(crop_path), crop_bgr)
        crop_records.append({
            "crop_index": crop_index,
            "source_path": str(source_path.relative_to(input_root)),
            "bbox_xyxy_resized": [x1, y1, x2, y2],
            "source_resize_scale": scale,
            "crop_path": str(crop_path),
        })
        if len(crop_records) >= TARGET_CROPS:
            break

detect_seconds = time.perf_counter() - detect_started
assert len(crop_records) == TARGET_CROPS, f"CRAFT found only {len(crop_records)} valid crops after scanning {scanned} real keyframes"
with MANIFEST_PATH.open("w", encoding="utf-8") as handle:
    for record in crop_records:
        handle.write(json.dumps(record, ensure_ascii=False) + "\n")
print({"real_crops": len(crop_records), "source_images_scanned": scanned, "craft_seconds": detect_seconds, "manifest": str(MANIFEST_PATH)})
del reader
torch.cuda.empty_cache()

In [ ]:
# Download the exact official revision, verify weight + remote-code blobs, then load locally as FP16.
from huggingface_hub import snapshot_download
from transformers import AutoModel, AutoTokenizer

run_started_utc = utc_now()
snapshot_started = time.perf_counter()
snapshot_dir = Path(snapshot_download(repo_id=MODEL_ID, revision=MODEL_REVISION))
snapshot_seconds = time.perf_counter() - snapshot_started
weight_path = snapshot_dir / "model.safetensors"
assert weight_path.is_file(), f"Missing {weight_path}"
assert weight_path.stat().st_size == MODEL_WEIGHT_BYTES, (weight_path.stat().st_size, MODEL_WEIGHT_BYTES)
actual_weight_sha256 = sha256_file(weight_path)
assert actual_weight_sha256 == MODEL_WEIGHT_SHA256, (actual_weight_sha256, MODEL_WEIGHT_SHA256)
actual_code_oids = {}
for filename, expected_oid in MODEL_REQUIRED_GIT_OIDS.items():
    source = snapshot_dir / filename
    assert source.is_file(), f"Pinned snapshot is missing {filename}"
    actual_oid = git_blob_oid(source)
    assert actual_oid == expected_oid, (filename, actual_oid, expected_oid)
    actual_code_oids[filename] = actual_oid

# The official config auto_map includes an explicit `repo_id--module` prefix.
# Transformers follows that prefix back to the network even when the same code is
# already present locally. Build a zero-copy runtime view and remove only that prefix.
# The downloaded pinned snapshot remains untouched and all source blobs were verified above.
runtime_dir = WORK_DIR / "vintern-local-runtime"
if runtime_dir.exists():
    shutil.rmtree(runtime_dir)
runtime_dir.mkdir(parents=True)
for source in snapshot_dir.iterdir():
    target = runtime_dir / source.name
    if source.name == "config.json":
        config_data = json.loads(source.read_text(encoding="utf-8"))
        config_data["auto_map"] = {key: value.split("--", 1)[-1] for key, value in config_data["auto_map"].items()}
        target.write_text(json.dumps(config_data, ensure_ascii=False, indent=2), encoding="utf-8")
    elif source.is_file():
        target.symlink_to(source.resolve())
assert (runtime_dir / "configuration_internvl_chat.py").is_file()
assert json.loads((runtime_dir / "config.json").read_text(encoding="utf-8"))["auto_map"]["AutoConfig"] == "configuration_internvl_chat.InternVLChatConfig"

torch.cuda.empty_cache()
torch.cuda.reset_peak_memory_stats(0)
free_before, total_bytes = torch.cuda.mem_get_info(0)
load_started = time.perf_counter()
tokenizer = AutoTokenizer.from_pretrained(str(runtime_dir), trust_remote_code=True, use_fast=False, local_files_only=True)
model = AutoModel.from_pretrained(
    str(runtime_dir),
    torch_dtype=torch.float16,
    low_cpu_mem_usage=True,
    trust_remote_code=True,
    local_files_only=True,
    use_flash_attn=False,
).eval().to("cuda:0")
torch.cuda.synchronize()
load_seconds = time.perf_counter() - load_started
free_after, _ = torch.cuda.mem_get_info(0)
load_stats = {
    "seconds": load_seconds,
    "used_delta_mib": (free_before - free_after) / 2**20,
    "allocated_mib": torch.cuda.memory_allocated(0) / 2**20,
    "reserved_mib": torch.cuda.memory_reserved(0) / 2**20,
    "peak_allocated_mib": torch.cuda.max_memory_allocated(0) / 2**20,
}
assert all(parameter.dtype == torch.float16 for parameter in model.parameters() if parameter.is_floating_point()), "Floating model parameters are not uniformly FP16"
print({"snapshot_seconds": snapshot_seconds, "weight_sha256": actual_weight_sha256, "remote_code_git_oids": actual_code_oids, "load": load_stats})

In [ ]:
# InternVL/Vintern image preprocessing from the official model family contract.
import torchvision.transforms as T
from torchvision.transforms.functional import InterpolationMode

IMAGENET_MEAN = (0.485, 0.456, 0.406)
IMAGENET_STD = (0.229, 0.224, 0.225)

def build_transform(input_size=448):
    return T.Compose([
        T.Lambda(lambda image: image.convert("RGB")),
        T.Resize((input_size, input_size), interpolation=InterpolationMode.BICUBIC),
        T.ToTensor(),
        T.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
    ])

def find_closest_aspect_ratio(aspect_ratio, target_ratios, width, height, image_size):
    best_ratio_diff = float("inf")
    best_ratio = (1, 1)
    area = width * height
    for ratio in target_ratios:
        target_aspect_ratio = ratio[0] / ratio[1]
        ratio_diff = abs(aspect_ratio - target_aspect_ratio)
        if ratio_diff < best_ratio_diff or (ratio_diff == best_ratio_diff and area > 0.5 * image_size * image_size * ratio[0] * ratio[1]):
            best_ratio_diff = ratio_diff
            best_ratio = ratio
    return best_ratio

def dynamic_preprocess(image, min_num=1, max_num=6, image_size=448, use_thumbnail=True):
    orig_width, orig_height = image.size
    aspect_ratio = orig_width / orig_height
    target_ratios = sorted({(i, j) for n in range(min_num, max_num + 1) for i in range(1, n + 1) for j in range(1, n + 1) if min_num <= i * j <= max_num}, key=lambda item: item[0] * item[1])
    target_ratio = find_closest_aspect_ratio(aspect_ratio, target_ratios, orig_width, orig_height, image_size)
    target_width = image_size * target_ratio[0]
    target_height = image_size * target_ratio[1]
    blocks = target_ratio[0] * target_ratio[1]
    resized = image.resize((target_width, target_height))
    images = []
    for index in range(blocks):
        box = (
            (index % (target_width // image_size)) * image_size,
            (index // (target_width // image_size)) * image_size,
            ((index % (target_width // image_size)) + 1) * image_size,
            ((index // (target_width // image_size)) + 1) * image_size,
        )
        images.append(resized.crop(box))
    if use_thumbnail and len(images) != 1:
        images.append(image.resize((image_size, image_size)))
    return images

transform = build_transform(448)
def load_crop(path, max_num):
    image = Image.open(path).convert("RGB")
    tiles = dynamic_preprocess(image, max_num=max_num, image_size=448, use_thumbnail=True)
    pixels = torch.stack([transform(tile) for tile in tiles]).to(device="cuda:0", dtype=torch.float16)
    return pixels, len(tiles)

In [ ]:
QUESTION = "<image>\nChép lại nguyên văn toàn bộ chữ nhìn thấy. Không sửa chính tả, không suy đoán phần bị che. Chỉ trả về văn bản."
GENERATION_CONFIG = {"max_new_tokens": MAX_NEW_TOKENS, "do_sample": False, "num_beams": 1}

def percentile(values, q):
    return float(np.percentile(np.asarray(values, dtype=np.float64), q))

mode_results = {}
sample_outputs = []
for max_num in TILE_MODES:
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats(0)
    latencies = []
    tile_counts = []
    failures = []
    # Warm-up is separate from the measured 100 crops.
    warm_pixels, _ = load_crop(crop_records[0]["crop_path"], max_num)
    with torch.inference_mode():
        _ = model.chat(tokenizer, warm_pixels, QUESTION, GENERATION_CONFIG)
    del warm_pixels
    torch.cuda.synchronize()
    for record in crop_records:
        try:
            pixels, tile_count = load_crop(record["crop_path"], max_num)
            torch.cuda.synchronize()
            started = time.perf_counter()
            with torch.inference_mode():
                response = model.chat(tokenizer, pixels, QUESTION, GENERATION_CONFIG)
            torch.cuda.synchronize()
            latency = time.perf_counter() - started
            latencies.append(latency)
            tile_counts.append(tile_count)
            if len(sample_outputs) < 12:
                sample_outputs.append({"max_num": max_num, "crop_index": record["crop_index"], "tiles": tile_count, "latency_seconds": latency, "text": str(response)[:500]})
            del pixels
        except torch.cuda.OutOfMemoryError as exc:
            failures.append({"crop_index": record["crop_index"], "error": "CUDA OOM", "detail": str(exc)[:500]})
            torch.cuda.empty_cache()
            break
        except Exception as exc:
            failures.append({"crop_index": record["crop_index"], "error": type(exc).__name__, "detail": str(exc)[:500]})
    mode_results[str(max_num)] = {
        "requested_crops": TARGET_CROPS,
        "successful_crops": len(latencies),
        "failures": failures,
        "tiles_min": min(tile_counts) if tile_counts else None,
        "tiles_max": max(tile_counts) if tile_counts else None,
        "tiles_mean": statistics.fmean(tile_counts) if tile_counts else None,
        "latency_seconds_mean": statistics.fmean(latencies) if latencies else None,
        "latency_seconds_p50": percentile(latencies, 50) if latencies else None,
        "latency_seconds_p95": percentile(latencies, 95) if latencies else None,
        "latency_seconds_max": max(latencies) if latencies else None,
        "throughput_crops_per_second": len(latencies) / sum(latencies) if latencies else None,
        "peak_allocated_mib": torch.cuda.max_memory_allocated(0) / 2**20,
        "peak_reserved_mib": torch.cuda.max_memory_reserved(0) / 2**20,
    }
    print("MODE_DONE", max_num, mode_results[str(max_num)])

all_pass = all(result["successful_crops"] == TARGET_CROPS and not result["failures"] for result in mode_results.values())
result = {
    "schema_version": 1,
    "run_started_utc": run_started_utc,
    "run_finished_utc": utc_now(),
    "decision": "PASS_FP16_T4" if all_pass else "NO_GO_FP16_T4",
    "model": {"id": MODEL_ID, "revision": MODEL_REVISION, "weight_bytes": MODEL_WEIGHT_BYTES, "weight_sha256_expected": MODEL_WEIGHT_SHA256, "weight_sha256_actual": actual_weight_sha256, "remote_code_git_oids": actual_code_oids, "runtime_auto_map_localized": True, "dtype": "float16", "trust_remote_code": True, "use_flash_attn": False},
    "environment": {"python": sys.version, "platform": platform.platform(), "visible_gpu": gpu_name, "visible_gpu_total_mib": total_bytes / 2**20, "physical_gpus_nvidia_smi": gpu_query(), "packages": package_versions()},
    "data": {"input_roots": [str(path) for path in all_roots], "source_images_scanned": scanned, "craft_real_crops": len(crop_records), "craft_seconds": detect_seconds, "craft_zip_sha256": CRAFT_ZIP_SHA256, "craft_weight_sha256": CRAFT_WEIGHT_SHA256, "crop_manifest": str(MANIFEST_PATH)},
    "download": {"snapshot_seconds": snapshot_seconds},
    "load": load_stats,
    "generation": {"prompt": QUESTION, **GENERATION_CONFIG},
    "tile_modes": mode_results,
    "sample_outputs": sample_outputs,
}
RESULT_PATH.write_text(json.dumps(result, ensure_ascii=False, indent=2), encoding="utf-8")
print("FINAL_RESULT_PATH", RESULT_PATH)
print(json.dumps(result, ensure_ascii=False, indent=2))
assert all_pass, "At least one FP16 T4 benchmark mode failed; inspect the JSON artifact"

## After the run

Download these two files from the Output panel and send them back for review:

- `vintern_t4_fp16_benchmark.json`
- `vintern_craft_crops_manifest.jsonl`

Then stop the Kaggle session. Do not use **Save Version** unless you intentionally want to publish a private notebook version.